# 03 — Voice verification

Owner: **Kudakwashe**

This notebook builds a speaker-identity classifier from original voice recordings. It uses SpeechBrain's pretrained [ECAPA-TDNN VoxCeleb model](https://huggingface.co/speechbrain/spkrec-ecapa-voxceleb) (Apache-2.0) for speaker embeddings. MFCCs, spectral roll-off and RMS are recorded as analysis features; the classifier uses only speaker embeddings.

Every original recording is assigned to one split before pitch-shift and noise variants are created. Augmented descendants stay with their original split. Validation samples calibrate the rejection threshold; test samples, when available, are never used to select it.

**Data limitation:** with only two original recordings per speaker, there is no independent test split. Validation/calibration metrics are provisional, not held-out performance. Record at least three independent sessions per enrolled person for train/validation/test evaluation; additional sessions are preferable. The unauthorized folder is currently empty, so unknown-person false-acceptance cannot be measured. This classroom model does not provide liveness or replay protection.

In [ ]:
from pathlib import Path
import sys
import zlib

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "main.py").exists(), "Run from the project root or notebooks directory"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
MODELS = ROOT / "models"

import librosa
import librosa.display
import matplotlib.pyplot as plt
import pandas as pd

from feature_extraction import load_voice_audio
from voice_model import (
    augment_audio,
    build_feature_table,
    discover_recordings,
    split_original_recordings,
    train_voice_model,
)


## 1. Inventory and split original recordings

The split is by original source and speaker. Two recordings per speaker produce train/validation only; the code does not claim an independent test result.

In [ ]:
recordings = discover_recordings(DATA / "raw" / "audio")
split_recordings = split_original_recordings(recordings, random_state=42)
inventory = pd.DataFrame(split_recordings)
display(inventory.groupby(["user_id", "split"], as_index=False).size())
print(f"Original recordings: {len(recordings)}; speakers: {inventory['user_id'].nunique()}")
if not (inventory["split"] == "test").any():
    print("No independent test split: each speaker currently has only two original recordings.")


## 2. Listen and inspect waveforms and spectrograms

The loader converts recordings to mono 16 kHz audio and trims leading/trailing silence at 30 dB below peak.

In [ ]:
for recording in split_recordings:
    audio, sample_rate = load_voice_audio(recording["path"])
    duration = len(audio) / sample_rate
    print(f"{recording['user_id']} / {recording['source_id']}: {duration:.2f}s at {sample_rate} Hz")
    figure, axes = plt.subplots(2, 1, figsize=(10, 6))
    librosa.display.waveshow(audio, sr=sample_rate, ax=axes[0])
    axes[0].set(title=f"Waveform — {recording['source_id']}", xlabel="Time (seconds)")
    spectrum = librosa.amplitude_to_db(abs(librosa.stft(audio)), ref=max)
    librosa.display.specshow(spectrum, sr=sample_rate, x_axis="time", y_axis="hz", ax=axes[1])
    axes[1].set(title="Spectrogram")
    figure.tight_layout()
    plt.show()


## 3. Apply two augmentations per original

The pitch-shifted and low-level-noise variants retain the source's split. Augmented validation/test rows are saved for provenance but are not used to fit the classifier or choose the threshold.

In [ ]:
for recording in split_recordings:
    audio, sample_rate = load_voice_audio(recording["path"])
    seed = zlib.crc32(recording["source_id"].encode("utf-8"))
    variants = [("original", audio)] + augment_audio(audio, sample_rate, seed=seed)
    figure, axes = plt.subplots(len(variants), 1, figsize=(10, 7), squeeze=False)
    for axis, (name, waveform) in zip(axes[:, 0], variants):
        librosa.display.waveshow(waveform, sr=sample_rate, ax=axis)
        axis.set(title=f"{recording['source_id']} — {name}", xlabel="Time (seconds)")
    figure.tight_layout()
    plt.show()


## 4. Extract features, train, and evaluate

The shared extractor returns normalized pretrained speaker embeddings. The CSV also contains MFCC, spectral roll-off and RMS summaries for analysis. Only embedding columns enter the classifier; provenance columns are never model inputs.

In [ ]:
feature_table = build_feature_table(split_recordings)
print(f"Feature rows: {len(feature_table)}")
display(feature_table.groupby(["user_id", "split", "augmentation"], as_index=False).size())
result = train_voice_model(
    feature_table,
    MODELS / "voice_model.joblib",
    DATA / "processed" / "audio_features.csv",
    MODELS / "model_metadata.json",
)
print(f"Provisional validation threshold: {result['threshold']:.4f}")
print("Validation/calibration metrics:", result["validation_metrics"])
print("Independent test metrics:", result["test_metrics"])
print(f"Saved voice model with {len(result['feature_columns'])} embedding features.")
print("Overall metadata status remains unchanged until all team models are trained and validated.")


## 5. Unknown-speaker check

Only recordings placed in `data/raw/audio/unauthorized/` are scored as unknown people. Different enrolled-speaker claims are included in validation FAR/FRR calculations. Do not copy another enrolled member's audio into the unknown folder.

In [ ]:
from feature_extraction import extract_voice_features

unauthorized_root = DATA / "raw" / "audio" / "unauthorized"
unauthorized_paths = sorted(
    path for path in unauthorized_root.rglob("*")
    if path.is_file() and path.suffix.lower() in {".flac", ".m4a", ".mp3", ".ogg", ".wav"}
)
unknown_false_accepts = 0
if not unauthorized_paths:
    print("No unknown-speaker recordings available; unknown-person FAR is not measured.")
else:
    for path in unauthorized_paths:
        probabilities = result["model"].predict_proba(extract_voice_features(path))
        best_index = int(probabilities[0].argmax())
        score = float(probabilities[0, best_index])
        decision = "accepted" if score >= result["threshold"] else "rejected"
        unknown_false_accepts += int(decision == "accepted")
        print(f"{path.name}: {decision} (best score={score:.4f})")
    unknown_far = unknown_false_accepts / len(unauthorized_paths)
    print(f"Unknown-speaker FAR: {unknown_far:.3f} ({unknown_false_accepts}/{len(unauthorized_paths)})")
